# EDA размеченных новостей

Исследование частичного датасета `data/labeling_dataset_labeled_partial.json` (36 000 размеченных записей). Ноутбук использует только стандартную библиотеку Python и IPython для красивого вывода текста.

In [15]:
import html
import json
import random
import re
import statistics
from collections import Counter
from pathlib import Path
from IPython.display import HTML, display

DATA_PATH = Path.cwd() / "data/labeling_dataset_labeled_partial.json"
if not DATA_PATH.exists():
    DATA_PATH = Path.cwd().parent / "data/labeling_dataset_labeled_partial.json"

with DATA_PATH.open(encoding="utf-8") as f:
    data = json.load(f)

assert data and all("text" in row and "target" in row for row in data)
print(f"Файл: {DATA_PATH.resolve()}\nЗаписей: {len(data):,}".replace(",", " "))

Файл: /Users/alexpechul/study/alfa-hackathon/ml/data/labeling_dataset_labeled_partial.json
Записей: 36 000


## Структура и качество данных

In [9]:
texts = [str(row.get("text") or "") for row in data]
targets = [row.get("target") or {} for row in data]
all_categories = sorted({category for target in targets for category in target})
active_categories = [[c for c, value in target.items() if value == 1] for target in targets]
label_counts = Counter(category for labels in active_categories for category in labels)
label_cardinality = [len(labels) for labels in active_categories]
empty_text_count = sum(not text.strip() for text in texts)
empty_target_count = sum(not labels for labels in active_categories)
unique_text_count = len(set(texts))

print(f"Категорий в схеме: {len(all_categories)}")
print(f"Пустой текст: {empty_text_count:,} ({empty_text_count / len(data):.2%})")
print(f"Без положительных меток: {empty_target_count:,} ({empty_target_count / len(data):.2%})")
print(f"Уникальных текстов: {unique_text_count:,} ({unique_text_count / len(data):.2%})")
print(f"Дубликаты текста сверх первого вхождения: {len(data) - unique_text_count:,}")
print(f"Среднее число меток на запись: {statistics.mean(label_cardinality):.3f}")
print(f"Записи с несколькими метками: {sum(n > 1 for n in label_cardinality):,}")
print("\nПример схемы target:")
display(HTML(f'<pre style="white-space:pre-wrap">{html.escape(json.dumps(data[0]["target"], ensure_ascii=False, indent=2))}</pre>'))

Категорий в схеме: 31
Пустой текст: 0 (0.00%)
Без положительных меток: 2,990 (8.31%)
Уникальных текстов: 35,624 (98.96%)
Дубликаты текста сверх первого вхождения: 376
Среднее число меток на запись: 1.663
Записи с несколькими метками: 17,807

Пример схемы target:


## Длина текстов

Считаем длину в символах и словах. Перцентили показывают, насколько длинные тексты попадают в хвост распределения.

In [10]:
word_counts = [len(re.findall(r"\S+", text)) for text in texts]
char_counts = [len(text) for text in texts]
percentiles = (0, 1, 5, 10, 25, 50, 75, 90, 95, 99, 100)

def percentile(values, p):
    ordered = sorted(values)
    position = (len(ordered) - 1) * p / 100
    lower = int(position)
    upper = min(lower + 1, len(ordered) - 1)
    fraction = position - lower
    return ordered[lower] * (1 - fraction) + ordered[upper] * fraction

def describe_lengths(values, title):
    print(title)
    print(f"Среднее: {statistics.mean(values):,.1f} | медиана: {statistics.median(values):,.1f} | min: {min(values):,} | max: {max(values):,}".replace(",", " "))
    print("Перцентиль | Значение")
    for p in percentiles:
        print(f"{p:>3}%        | {percentile(values, p):,.1f}".replace(",", " "))

describe_lengths(char_counts, "Длина в символах")
print()
describe_lengths(word_counts, "Длина в словах")

Длина в символах
Среднее: 926.7 | медиана: 723.0 | min: 3 | max: 8 175
Перцентиль | Значение
  0%        | 3.0
  1%        | 54.0
  5%        | 113.0
 10%        | 169.0
 25%        | 340.0
 50%        | 723.0
 75%        | 1 202.0
 90%        | 1 995.1
 95%        | 2 674.1
 99%        | 3 935.0
100%        | 8 175.0

Длина в словах
Среднее: 127.8 | медиана: 100.0 | min: 1 | max: 1 185
Перцентиль | Значение
  0%        | 1.0
  1%        | 7.0
  5%        | 16.0
 10%        | 22.0
 25%        | 47.0
 50%        | 100.0
 75%        | 165.0
 90%        | 273.0
 95%        | 371.0
 99%        | 541.0
100%        | 1 185.0


In [4]:
def histogram(values, bins=12):
    low, high = min(values), max(values)
    if low == high:
        return [(low, high, len(values))]
    width = (high - low) / bins
    counts = [0] * bins
    for value in values:
        counts[min(int((value - low) / width), bins - 1)] += 1
    return [(low + i * width, low + (i + 1) * width, count) for i, count in enumerate(counts)]

def show_histogram(values, title, bins=12):
    buckets = histogram(values, bins)
    max_count = max(count for _, _, count in buckets)
    print(title)
    for start, end, count in buckets:
        bar = "█" * round(36 * count / max_count)
        print(f"{start:>9,.0f}–{end:<9,.0f} {bar:<36} {count:>6,}".replace(",", " "))

show_histogram(word_counts, "Распределение длины (слов)")

Распределение длины (слов)
        1–100       ████████████████████████████████████ 17 952
      100–198       ███████████████████████              11 467
      198–297       ███████                               3 529
      297–396       ███                                   1 580
      396–494       ██                                      804
      494–593       █                                       531
      593–692                                               126
      692–790                                                 7
      790–889                                                 1
      889–988                                                 0
      988–1 086                                               1
    1 086–1 185                                               2


## Частоты категорий

Разметка multi-label: сумма по категориям может быть больше числа новостей.

In [5]:
category_rows = sorted(label_counts.items(), key=lambda item: (-item[1], item[0]))
max_count = max(label_counts.values(), default=1)
print(f"{'Категория':48} {'Новостей':>9} {'Доля':>8}  Частота")
print("-" * 100)
for category, count in category_rows:
    bar = "█" * round(28 * count / max_count)
    print(f"{category[:48]:48} {count:>9,} {count / len(data):>7.1%}  {bar}".replace(",", " "))

print(f"\nВсего положительных присвоений меток: {sum(label_counts.values()):,}".replace(",", " "))

Категория                                         Новостей     Доля  Частота
----------------------------------------------------------------------------------------------------
Геополитика и мировая экономика                      6 307   17.5%  ████████████████████████████
IT  телеком и ИИ                                     3 944   11.0%  ██████████████████
Логистика и перевозки                                3 278    9.1%  ███████████████
Нефтегаз и ТЭК                                       2 900    8.1%  █████████████
Реклама/мусор                                        2 745    7.6%  ████████████
Банки и финтех                                       2 647    7.4%  ████████████
АПК и продовольствие                                 2 594    7.2%  ████████████
Розничная торговля и e-commerce                      2 542    7.1%  ███████████
Налоги и налоговое администрирование                 2 256    6.3%  ██████████
Санкции и экспортный контроль                        2 142    5.9%  ██

In [6]:
cardinality_counts = Counter(label_cardinality)
print("Распределение числа меток на новость:")
for n in range(max(cardinality_counts, default=0) + 1):
    count = cardinality_counts[n]
    print(f"{n:>2} меток: {count:>7,} ({count / len(data):6.2%})".replace(",", " "))

pair_counts = Counter()
for labels in active_categories:
    labels = sorted(set(labels))
    for i, first in enumerate(labels):
        for second in labels[i + 1:]:
            pair_counts[(first, second)] += 1

print("\nСамые частые пары категорий:")
for (first, second), count in pair_counts.most_common(15):
    print(f"{count:>6,}  {first}  +  {second}".replace(",", " "))

Распределение числа меток на новость:
 0 меток:   2 990 ( 8.31%)
 1 меток:  15 203 (42.23%)
 2 меток:  12 070 (33.53%)
 3 меток:   3 998 (11.11%)
 4 меток:   1 088 ( 3.02%)
 5 меток:     313 ( 0.87%)
 6 меток:     146 ( 0.41%)
 7 меток:      63 ( 0.18%)
 8 меток:      43 ( 0.12%)
 9 меток:      26 ( 0.07%)
10 меток:      20 ( 0.06%)
11 меток:      11 ( 0.03%)
12 меток:      10 ( 0.03%)
13 меток:       6 ( 0.02%)
14 меток:       6 ( 0.02%)
15 меток:       3 ( 0.01%)
16 меток:       0 ( 0.00%)
17 меток:       0 ( 0.00%)
18 меток:       1 ( 0.00%)
19 меток:       2 ( 0.01%)
20 меток:       0 ( 0.00%)
21 меток:       1 ( 0.00%)

Самые частые пары категорий:
 1 551  Геополитика и мировая экономика  +  Нефтегаз и ТЭК
 1 247  Геополитика и мировая экономика  +  Санкции и экспортный контроль
   821  Нефтегаз и ТЭК  +  Санкции и экспортный контроль
   746  Геополитика и мировая экономика  +  Логистика и перевозки
   611  Геополитика и мировая экономика  +  Металлы  руды и угли
   574  АПК и про

## Примеры новостей по категориям

Для каждой выбранной категории показываем один пример с непустой меткой. HTML-карточки сохраняют переносы строк и ограничивают ширину текста для удобного чтения.

In [7]:
def render_news(index, max_chars=5000):
    text = texts[index]
    if len(text) > max_chars:
        text = text[:max_chars] + "\n\n[текст сокращён для показа]"
    labels = active_categories[index]
    safe_labels = " · ".join(html.escape(label) for label in labels) or "Без меток"
    body = html.escape(text)
    return (
        '<article style="max-width:900px;margin:16px 0;padding:18px 22px;'
        'border:1px solid #d8dee4;border-left:4px solid #287a68;border-radius:6px;'
        'font:15px/1.65 -apple-system,BlinkMacSystemFont,Segoe UI,sans-serif;'
        'color:#24292f;background:#fff">'
        f'<div style="font-size:12px;font-weight:600;color:#287a68;margin-bottom:10px">{safe_labels}</div>'
        f'<div style="white-space:pre-wrap;overflow-wrap:anywhere">{body}</div>'
        f'<div style="font-size:11px;color:#6e7781;margin-top:12px">Запись #{index:,} · {len(texts[index]):,} символов</div>'
        '</article>'
    )

sample_categories = [category for category, _ in category_rows[:10]]
cards = []
for category in sample_categories:
    index = next((i for i, labels in enumerate(active_categories) if category in labels), None)
    if index is not None:
        cards.append(render_news(index))
display(HTML("".join(cards)))

## Короткие и длинные тексты

Сравним несколько крайних записей. Очень короткие новости и длинные посты удобно проверять вручную на шум, обрезку и нетипичный формат.

In [8]:
nonempty_indices = [i for i, text in enumerate(texts) if text.strip()]
shortest = sorted(nonempty_indices, key=lambda i: word_counts[i])[:3]
longest = sorted(nonempty_indices, key=lambda i: word_counts[i], reverse=True)[:3]
print("Самые короткие:")
display(HTML("".join(render_news(i, max_chars=1200) for i in shortest)))
print("Самые длинные:")
display(HTML("".join(render_news(i, max_chars=5000) for i in longest)))

Самые короткие:


Самые длинные:


## Примеры новостей без меток

Показываем несколько записей, для которых все значения `target` равны нулю.

In [18]:
unlabeled_indices = [i for i, labels in enumerate(active_categories) if not labels]
unlabeled_sample = random.sample(unlabeled_indices, min(10, len(unlabeled_indices)))
print(f"Новостей без положительных меток: {len(unlabeled_indices):,}; показано: {len(unlabeled_sample)}".replace(",", " "))
display(HTML("".join(render_news(i) for i in unlabeled_sample)))

Новостей без положительных меток: 2 990; показано: 10


## Примеры с меткой «Реклама/мусор»

Каждый запуск ячейки выбирает до 10 случайных новостей с этой меткой.

In [22]:
ad_label = "Страхование"
ad_indices = [i for i, target in enumerate(targets) if target.get(ad_label) == 1]
ad_sample = random.sample(ad_indices, min(10, len(ad_indices)))
print(f"Новостей с меткой «{ad_label}»: {len(ad_indices):,}; показано: {len(ad_sample)}".replace(",", " "))
display(HTML("".join(render_news(i) for i in ad_sample)))

Новостей с меткой «Страхование»: 1 933; показано: 10
